In [1]:
import torch
import pandas as pd

from src.model import MultimodalFakeNewsDetectionModel
from src.dataloader import create_dataloader

W0717 19:57:16.450000 1084 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


CUDA Available: True


In [2]:
print(hasattr(MultimodalFakeNewsDetectionModel, "test_epoch_end"))

False


In [13]:
print("PyTorch Version :", torch.__version__)
print("CUDA Available  :", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using Device :", device)

if device.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

PyTorch Version : 2.13.0+cu126
CUDA Available  : True
Using Device : cuda
GPU : NVIDIA GeForce RTX 2050


In [14]:
BATCH_SIZE = 32
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4

In [15]:


train_embeddings = torch.load(
    "./data/embeddings/train_embeddings.pt"
)

validation_embeddings = torch.load(
    "./data/embeddings/validation_embeddings.pt"
)

test_embeddings = torch.load(
    "./data/embeddings/test_embeddings.pt"
)

train_loader = create_dataloader(
    csv_path="./data/processed/train_clean.csv",
    embeddings=train_embeddings,
    shuffle=True,
)

val_loader = create_dataloader(
    csv_path="./data/processed/validation_clean.csv",
    embeddings=validation_embeddings,
    shuffle=False,
)
test_loader = create_dataloader(
    csv_path="./data/processed/test_clean.csv",
    embeddings=test_embeddings,
    shuffle=False,
)

In [16]:
sample = next(iter(train_loader))

print(sample["text"].shape)
print(sample["image"].shape)
print(sample["label"].shape)

torch.Size([32, 768])
torch.Size([32, 3, 224, 224])
torch.Size([32])


In [17]:
print(len(train_embeddings))

31296


In [18]:
batch = next(iter(train_loader))

print(batch.keys())

dict_keys(['id', 'text', 'image', 'label'])


In [19]:
model = MultimodalFakeNewsDetectionModel()

print(model)

MultimodalFakeNewsDetectionModel(
  (model): JointTextImageModel(
    (text_module): Linear(in_features=768, out_features=300, bias=True)
    (image_module): ResNet(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [20]:
model = model.to(device)

print(next(model.parameters()).device)

cuda:0


In [21]:
optimizer = model.configure_optimizers()

print(optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0
)


In [22]:
criterion = torch.nn.CrossEntropyLoss()

print(criterion)

CrossEntropyLoss()


In [23]:
batch = next(iter(train_loader))

text = batch["text"].to(device)
image = batch["image"].to(device)
label = batch["label"].to(device)

model.eval()

with torch.no_grad():
    pred, loss = model(text, image, label)

print("Prediction shape:", pred.shape)
print("Loss:", loss.item())

Prediction shape: torch.Size([32, 2])
Loss: 0.6900514364242554


In [24]:
import torch
import pytorch_lightning as pl

from pytorch_lightning.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
)

In [25]:
checkpoint_callback = ModelCheckpoint(
    monitor="val_acc",
    mode="max",
    save_top_k=1,
    filename="best-model",
)

In [26]:
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=3,
    mode="min",
)

In [27]:
trainer = pl.Trainer(
    max_epochs=10,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    callbacks=[
        checkpoint_callback,
        early_stop_callback,
    ],
    log_every_n_steps=10,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [10]:
trainer.fit(
    model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
)

NameError: name 'model' is not defined

In [28]:
best_model = MultimodalFakeNewsDetectionModel.load_from_checkpoint(
    "lightning_logs/version_0/checkpoints/best-model.ckpt"
)

In [29]:
trainer.test(
    model=best_model,
    dataloaders=test_loader,
)

You are using a CUDA device ('NVIDIA GeForce RTX 2050') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

c:\Users\suraj\AppData\Local\Programs\Python\Python311\Lib\site-packages\PIL\Image.py:1136: UserWarning: Palette 
images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(

c:\Users\suraj\AppData\Local\Programs\Python\Python311\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\suraj\AppData\Local\Programs\Python\Python311\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.8129478096961975     │
│         test_loss         │    0.4153093695640564     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.4153093695640564, 'test_acc': 0.8129478096961975}]

In [ ]:
import torch

best_model.eval()

y_true = []
y_pred = []
y_prob = []

device = next(best_model.parameters()).device

with torch.no_grad():
    for batch in test_loader:

        text = batch["text"].to(device)
        image = batch["image"].to(device)
        labels = batch["label"].to(device)

        outputs, _ = best_model.model(text, image, labels)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs[:, 1].cpu().numpy())   # probability of class 1